In [ ]:
# robust fairness (city n >= 50)

def filter_groups_by_min_n(df, group_col, min_n=50):
    counts = df[group_col].value_counts(dropna=False)
    keep = counts[counts >= min_n].index
    df_f = df[df[group_col].isin(keep)].copy()
    return df_f, counts, keep

In [ ]:
# sqrt reweighting (1 / sqrt(n))

import numpy as np

def compute_sqrt_inverse_weights(city_series):
    city_counts = city_series.value_counts()
    raw_w = 1.0 / np.sqrt(city_counts)
    norm_w = raw_w / raw_w.mean()
    return norm_w

In [ ]:
# fairness helper functions (gap/tpr/fpr)

import numpy as np
import pandas as pd


def gap_by_group(df, group_col):
    """
    Accuracy gap:
    max_group_accuracy - min_group_accuracy
    """
    group_acc = (
        df.groupby(group_col)["correct"]
        .mean()
        .dropna()
    )
    
    if len(group_acc) == 0:
        return np.nan
    
    return group_acc.max() - group_acc.min()


def ovr_rates_by_group(df, group_col, num_classes):
    """
    Computes one-vs-rest TPR and FPR for each class and group.

    Returns:
        tpr: array (num_groups, num_classes)
        fpr: array (num_groups, num_classes)
        support_pos: number of positive samples per (group, class)
        support_neg: number of negative samples per (group, class)
    """
    
    groups = sorted(df[group_col].dropna().unique())
    group_index = {g: i for i, g in enumerate(groups)}
    
    tpr = np.zeros((len(groups), num_classes))
    fpr = np.zeros((len(groups), num_classes))
    support_pos = np.zeros((len(groups), num_classes))
    support_neg = np.zeros((len(groups), num_classes))
    
    for g in groups:
        df_g = df[df[group_col] == g]
        gi = group_index[g]
        
        y_true = df_g["y_true"].values
        y_pred = df_g["y_pred"].values
        
        for c in range(num_classes):
            # Positive = class c
            pos_mask = (y_true == c)
            neg_mask = (y_true != c)
            
            TP = np.sum((y_pred == c) & pos_mask)
            FP = np.sum((y_pred == c) & neg_mask)
            FN = np.sum((y_pred != c) & pos_mask)
            TN = np.sum((y_pred != c) & neg_mask)
            
            support_pos[gi, c] = np.sum(pos_mask)
            support_neg[gi, c] = np.sum(neg_mask)
            
            # TPR = TP / (TP + FN)
            if (TP + FN) > 0:
                tpr[gi, c] = TP / (TP + FN)
            else:
                tpr[gi, c] = np.nan
            
            # FPR = FP / (FP + TN)
            if (FP + TN) > 0:
                fpr[gi, c] = FP / (FP + TN)
            else:
                fpr[gi, c] = np.nan
    
    return tpr, fpr, support_pos, support_neg


def summarize_gaps(rate_matrix):
    """
    Given rate matrix (num_groups, num_classes),
    computes:
        - gap per class
        - worst class gap
        - macro gap (mean over classes)
    """
    
    gap_per_class = []
    
    for c in range(rate_matrix.shape[1]):
        col = rate_matrix[:, c]
        col = col[~np.isnan(col)]
        
        if len(col) == 0:
            gap_per_class.append(np.nan)
        else:
            gap_per_class.append(np.max(col) - np.min(col))
    
    gap_per_class = np.array(gap_per_class)
    
    valid = gap_per_class[~np.isnan(gap_per_class)]
    
    if len(valid) == 0:
        return gap_per_class, np.nan, np.nan
    
    worst_gap = np.max(valid)
    macro_gap = np.mean(valid)
    
    return gap_per_class, worst_gap, macro_gap

In [ ]:
# unified evaluation function

from sklearn.metrics import accuracy_score, f1_score

def evaluate_all(df_test, y_true, y_pred, num_classes, min_city_n=50):
    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    df_eval["correct"] = (df_eval["y_true"] == df_eval["y_pred"])

    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))

    # Accuracy gaps
    out["gap_acc_gender"] = float(gap_by_group(df_eval, "gender"))
    out["gap_acc_age"]    = float(gap_by_group(df_eval, "age_group"))
    out["gap_acc_city"]   = float(gap_by_group(df_eval, "city_group"))

    # OVR gaps (full)
    for col in ["gender", "age_group", "city_group"]:
        tpr, fpr, _, _ = ovr_rates_by_group(df_eval, col, num_classes)
        _, tpr_worst, tpr_macro = summarize_gaps(tpr)
        _, fpr_worst, fpr_macro = summarize_gaps(fpr)
        out[f"{col}_tpr_gap_macro"] = tpr_macro
        out[f"{col}_tpr_gap_worst"] = tpr_worst
        out[f"{col}_fpr_gap_macro"] = fpr_macro
        out[f"{col}_fpr_gap_worst"] = fpr_worst

    # Robust city only
    df_city_rob, city_counts, keep = filter_groups_by_min_n(df_eval, "city_group", min_city_n)
    tpr_r, fpr_r, _, _ = ovr_rates_by_group(df_city_rob, "city_group", num_classes)
    _, tpr_worst_r, tpr_macro_r = summarize_gaps(tpr_r)
    _, fpr_worst_r, fpr_macro_r = summarize_gaps(fpr_r)

    out["city_robust_min_n"] = int(min_city_n)
    out["city_robust_num_groups"] = int(len(keep))
    out["city_robust_rows"] = int(len(df_city_rob))
    out["city_robust_tpr_gap_macro"] = tpr_macro_r
    out["city_robust_tpr_gap_worst"] = tpr_worst_r
    out["city_robust_fpr_gap_macro"] = fpr_macro_r
    out["city_robust_fpr_gap_worst"] = fpr_worst_r

    return out

In [ ]:
# final comparison table (baseline vs sqrt rw)

MIN_CITY_N = 50

res_base = evaluate_all(df_test, y_true_base, y_pred_base, num_labels, min_city_n=MIN_CITY_N)
res_rw1  = evaluate_all(df_test, y_true_rw1,  y_pred_rw1,  num_labels, min_city_n=MIN_CITY_N)
res_rw2  = evaluate_all(df_test, y_true_rw2,  y_pred_rw2,  num_labels, min_city_n=MIN_CITY_N)

results = pd.DataFrame([
    {"model": "baseline_1ep", **res_base},
    {"model": "sqrt_rw_1ep",  **res_rw1},
    {"model": "sqrt_rw_2ep",  **res_rw2},
])

results

In [ ]:
# worst-case city × class diagnostics (for final model sqrt_rw_2ep)

df_eval = df_test.copy()
df_eval["y_true"] = y_true_rw2
df_eval["y_pred"] = y_pred_rw2

rows = []

for c in range(num_labels):

    df_c = df_eval.copy()

    df_c["is_pos"] = (df_c["y_true"] == c).astype(int)
    df_c["is_tp"] = ((df_c["y_true"] == c) & (df_c["y_pred"] == c)).astype(int)

    stats = df_c.groupby("city_group").agg(
        positives=("is_pos", "sum"),
        tp=("is_tp", "sum")
    )

    stats["tpr"] = stats["tp"] / stats["positives"]

    # минимальный support по классу
    stats = stats[stats["positives"] >= 5]

    if len(stats) == 0:
        continue

    worst_city = stats["tpr"].idxmin()
    best_city = stats["tpr"].idxmax()

    rows.append({
        "class_id": c,
        "class_name": le.inverse_transform([c])[0],
        "worst_city": worst_city,
        "best_city": best_city,
        "TPR_worst": stats.loc[worst_city, "tpr"],
        "TPR_best": stats.loc[best_city, "tpr"],
        "support_worst": stats.loc[worst_city, "positives"],
        "support_best": stats.loc[best_city, "positives"]
    })

worst_case_table = pd.DataFrame(rows)

worst_case_table["tpr_gap"] = (
    worst_case_table["TPR_best"] - worst_case_table["TPR_worst"]
)

worst_case_table = worst_case_table.sort_values(
    "tpr_gap",
    ascending=False
)

worst_case_table

In [ ]:
# save worst-case TPR gap analysis by city group

worst_case_table.to_csv(
    "../figures/city_worst_case_analysis.csv",
    index=False
)

In [ ]:
# filter robust cases

robust_table = worst_case_table[
    worst_case_table["support_worst"] >= 30
]

robust_table

In [ ]:
# label prediction errors for frontend

frontend_class = le.transform(["web_frontend"])[0]

df_front = df_test.copy()
df_front["y_true"] = y_true_rw2
df_front["y_pred"] = y_pred_rw2

df_front = df_front[
    (df_front["y_true"] == frontend_class) &
    (df_front["city_group"] == "Москва")
]

df_front["correct"] = df_front["y_true"] == df_front["y_pred"]

df_front["correct"].value_counts()

In [ ]:
# analyze frontend misclassification examples
errors_front = df_front[df_front["correct"] == False]

errors_front[["resume_text", "y_pred"]].head(10)

In [ ]:
# most common error words

from collections import Counter
import re

texts = errors_front["resume_text"].tolist()

words = []

for t in texts:
    tokens = re.findall(r"\b\w+\b", t.lower())
    words.extend(tokens)

top_words = Counter(words).most_common(30)

top_words

In [ ]:
# frontend error word analysis cell

df_eval = df_test.copy()
df_eval["y_true"] = y_true_rw2
df_eval["y_pred"] = y_pred_rw2

rows = []

for c in range(num_labels):

    df_c = df_eval.copy()

    df_c["is_pos"] = (df_c["y_true"] == c).astype(int)
    df_c["is_tp"] = ((df_c["y_true"] == c) & (df_c["y_pred"] == c)).astype(int)

    stats = df_c.groupby("city_group").agg(
        positives=("is_pos", "sum"),
        tp=("is_tp", "sum")
    )

    stats["tpr"] = stats["tp"] / stats["positives"]

    stats = stats.reset_index()
    stats["class"] = le.inverse_transform([c])[0]

    rows.append(stats)

heatmap_df = pd.concat(rows)

In [ ]:
# tpr pivot by city class

pivot = heatmap_df.pivot(
    index="city_group",
    columns="class",
    values="tpr"
)

In [ ]:
# city class tpr heatmap visualization

import sys
import subprocess

def install_and_import(package):
    import importlib
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        print(f"Module '{package}' not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    finally:
        globals()[package] = importlib.import_module(package)

install_and_import("matplotlib")
install_and_import("numpy")

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12,6))

plt.imshow(pivot, aspect="auto")

plt.colorbar(label="TPR")

plt.xticks(
    np.arange(len(pivot.columns)),
    pivot.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(len(pivot.index)),
    pivot.index
)

plt.title("TPR by City and Class")

plt.tight_layout()
plt.show()

In [ ]:
# save city class tpr heatmap
plt.savefig("../figures/city_class_tpr_heatmap.png", dpi=300)